In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os 
load_dotenv()
if os.environ["GROQ_API_KEY"]:
    print("load env variable")
else:
    raise ValueError("not loaded")

load env variable


In [3]:
from langchain_core.messages import HumanMessage
llm = ChatGroq(
    model_name="qwen/qwen3.6-27b",
    temperature=0.7
)

llm.invoke([HumanMessage(content="I want to know the meaning of water")]).content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - **Query:** "I want to know the meaning of water"\n   - **Key Concept:** "water"\n   - **Intent:** The user is asking for the "meaning" of water. This is a bit ambiguous. It could mean:\n     - Scientific/chemical definition\n     - Biological/ecological importance\n     - Cultural/symbolic/spiritual meanings\n     - Linguistic/etymological meaning\n     - Philosophical meaning\n   - **Goal:** Provide a comprehensive but clear response that covers the most likely interpretations while asking for clarification if needed.\n\n2.  **Identify Key Dimensions of "Meaning of Water":**\n   - **Scientific/Chemical:** H₂O, liquid at room temperature, universal solvent, essential for life.\n   - **Biological/Ecological:** Vital for all known life forms, regulates climate, drives weather cycles, supports ecosystems.\n   - **Cultural/Symbolic:** Purity, life, renewal, emotion, subconscious, transition (in various religions/m

In [4]:
from typing import TypedDict,List,Annotated
from operator import add
class graph_schema (TypedDict):
        messages_manual: List
    #Using Reducer in Langgraph
        messages_auto: Annotated[List,add]

In [ ]:
from langchain_core.messages import AIMessage
def create_post(state: graph_schema) -> graph_schema:
    # 1. Manual track (no reducer in schema, handle appending manually)
    messages_manual = state['messages_manual']
    response_manual = llm.invoke(messages_manual).content 
    response_manual_ai = AIMessage(content=response_manual)

    # 2. Auto track (reducer enabled, DO NOT mutate state in place)
    messages_auto = state['messages_auto']
    response_auto = llm.invoke(messages_auto).content
    response_auto_ai = AIMessage(content=response_auto)

    return {
        "messages_manual": messages_manual + [response_manual_ai],
        "messages_auto": [response_auto_ai]  # Reducer adds this to existing list
    }


def curate_post(state: graph_schema) -> graph_schema:
    # 1. Manual track
    messages_manual = state['messages_manual']s
    response_manual = llm.invoke(messages_manual).content
    response_manual_ai = AIMessage(content=response_manual)

    # 2. Auto track
    messages_auto = state['messages_auto']
    response_auto = llm.invoke(messages_auto).content
    response_auto_ai = AIMessage(content=response_auto)

    return {
        "messages_manual": messages_manual + [response_manual_ai],
        "messages_auto": [response_auto_ai]  # Reducer adds this to existing list
    }

In [14]:
from langgraph.graph import StateGraph, START, END
graph = StateGraph(graph_schema)

graph.add_node("create_post", create_post)
graph.add_node("curate_post", curate_post)

graph.add_edge(START, "create_post")
graph.add_edge("create_post", "curate_post")
graph.add_edge("curate_post", END)

messages_graph = graph.compile()

# Execution
result = messages_graph.invoke(
    {
        "messages_manual": [HumanMessage(content="The importance of data privacy in the digital age")],
        "messages_auto": [HumanMessage(content="The importance of data privacy in the digital age")]
    }
)

print(result)

{'messages_manual': [HumanMessage(content='The importance of data privacy in the digital age', additional_kwargs={}, response_metadata={}), AIMessage(content='\n<think>\nHere\'s a thinking thinking sequence\n\n1.  **Deconstruct the user\'s query:**\n    *   Topic: "The importance of data privacy in the digital age."\n    *   Key concepts: Data privacy, digital age, importance.\n\n2.  **Initial analysis and brainstorming:**\n    *   *What is data privacy?* It\'s about controlling who has access to personal information.\n    *   *What is the digital age?* Everything is online, connected, data-driven, AI-powered, IoT, cloud computing.\n    *   *Why is it important?* Security, autonomy, reputation, legal compliance, economic value, trust.\n    *   *What are the threats?* Data breaches, identity theft, surveillance, manipulation, discrimination.\n    *   *Who is affected?* Individuals, businesses, governments.\n\n3.  **Structuring the response:**\n    *   A comprehensive answer needs struct